# Interrogación 1 - Métricas de Recomendación

La evaluación constará de una parte teórica y una segunda parte práctica donde deberás implementar con código en Python la métrica solicitada.

`NOMBRE ESTUDIANTE`: < colocar_nombre >

## Preguntas Teóricas (3pts)

En esta sección deberás contestar 3 preguntas teóricas

1. Explique brevemente las métricas MAP (Mean Average Precision) y AUC (ROC Area Under the Curve). Luego, mencione en que se diferencian ambas métricas, detallando el rango de valores de cada una, la interpretación de sus resultados y la forma de calcularlas.

**Respuesta**: MAP mide la calidad del ranking de los ítems relevantes en las listas de recomendación, promediando la precisión en todas las posiciones en las que aparece un ítem relevante y luego promediando entre todos los usuarios. AUC mide la capacidad del sistema de recomendación para distinguir entre ítems relevantes y no relevantes al comparar pares de ítems. Equivale a la probabilidad de que un ítem relevante tenga un puntaje mayor que un ítem no relevante.

MAP presenta valores entre 0 y 1, mientras que AUC toma valores entre 0.5 y 1.

MAP refleja qué tan pronto y consistentemente aparecen los ítems relevantes en la lista recomendada. Premia más cuando los relevantes están en los primeros lugares. Por su parte, AUC mide el poder de discriminación global del ranking, sin importar la posición exacta de los ítems relevantes en la lista.

El cálculo de MAP se realiza sumando las precisiones acumuladas en las posiciones donde aparece un ítem relevante, y se divide por el número total de relevantes de ese usuario. AUC toma todos los pares de ítems relevantes y no relevantes, se cuenta la proporción de veces en que el relevante está rankeado por encima del no relevante y se promedia sobre todos los usuarios.

2. Explique brevemente la métrica nDCG (normalized Discounted Cummulative Gain). ¿Qué se requiere para poder calcularla? ¿Qué diferencia tiene con DCG? ¿Qué fortalezas tiene nDCG en relación a otras métricas de ranking vistas en el curso?

**Respuesta**: Para calcular nDCG se requiere la lista de recomendación para un usuario, saber que items son relevantes, y haber calculado el DCG y el iDCG.

DCG mide la ganancia acumulada con descuento por posición, pero su valor depende de la longitud de la lista y del número de relevantes, mientras que nDCG corresponde a la versión normalizada dividiendo por el iDCG, lo que lleva todos los valores a un rango entre 0 y 1. Así se puede comparar entre usuarios, listas y sistemas distintos.

nDCG tiene como fortaleza considerar grados de relevancia de los items, y un decaimiento controlado de la penalización que se da a un item relevante según su posición. Además, al hacer la comparación con iDCG facilita entender la calidad del ranking en términos relativos al ideal.

3. Explica con tus palabras cuál es la diferencia entre RMSE y MAE, tanto en la forma en que se calculan como en la interpretación que entregan. ¿En qué situaciones podría ser más apropiado usar cada una para evaluar un modelo de recomendación?

**Respuesta:** MAE mide en promedio cuánto se equivoca el modelo, y se calcula en base a la siguiente fórmula:

$$
\text{MAE} = \frac{\displaystyle \sum_{i=1}^n |\hat{r}_{ui} - r_{ui}|}{n}
$$

Por su parte, RMSE calcula la raíz cuadradad de los errores al cuadrado, y se calcula de la siguiente manera:

$$
\text{RMSE} = \sqrt{\frac{\displaystyle \sum_{i=1}^n (\hat{r}_{ui} - r_{ui})^2}{n}}
$$

MAE puede ser más apropiado para casos en que nos importa la precisión promedio de las predicciones sin castigar demasiado a los casos extremos, mientras que RMSE es útil cuando nos importa evitar predicciones muy alejadas del valor real porque pueden ser críticas para la experiencia del usuario.


## Implementeación de Métrica (7 pts)

En esta sección deberás implementar la métrica AUC (ROC Area Under the Curve). Para hacer la evaluación de esta métrica vamos a utilizar el algoritmo de *collaborative filtering* denominado **Sapling Similarity**, presentado en el paper [Sapling Similarity: a performing and interpretable memory-based tool for recommendation](https://arxiv.org/abs/2210.07039), aplicado al dataset MovieLens 100K. Este dataset contiene 100.000 tuplas (usuario, item, ranking), de las cuales 80.000 se utilizarán para entrenamiento y 20.000 para testear. Los rankings se encuentran en un rango de 1 a 5.

Contarán con dos instancias del modelo ya entrenados: Basada en Usuario y Basada en Items. No deben modificar el código del modelo ni del entrenamiento.

In [ ]:
from sapling_similarity import SaplingSimilarity
import numpy as np

In [ ]:
# modelo basado en usuarios
sapling_similarity_users_model = SaplingSimilarity.from_csv(
    train_csv="train.csv",
    test_csv="test.csv",
    user_column_name="user_id",
    item_column_name="movie_id",
    rating_column_name="rating",
    threshold=3,
    based_on="user"
)

sapling_similarity_users_model.train()

# modelo basado en items
sapling_similarity_items_model = SaplingSimilarity.from_csv(
    train_csv="train.csv",
    test_csv="test.csv",
    user_column_name="user_id",
    item_column_name="movie_id",
    rating_column_name="rating",
    threshold=3,
    based_on="item"
)

sapling_similarity_items_model.train()

/content/sapling_similarity.py:129: RuntimeWarning: invalid value encountered in divide
  B = np.nan_to_num((1 - (co_ocurrences_matrix * (1 - co_ocurrences_matrix / items_interactions) + (items_interactions - co_ocurrences_matrix.T).T * (1 - (items_interactions - co_ocurrences_matrix.T).T / (
/content/sapling_similarity.py:130: RuntimeWarning: invalid value encountered in divide
  number_of_items - items_interactions))).T / (items_interactions * (1 - items_interactions / number_of_items))).T * np.sign(((co_ocurrences_matrix * number_of_items / items_interactions).T / items_interactions).T - 1))


Los modelos entrenados cuenta con el método `test`, el cual retorna dos elementos:
- El primer elemento corresponde a una lista de tuplas con los **ratings reales** para las combinaciones usuarios e items presentes en el set de testeo, con la forma (user_id, item_id, rating_real).

- El segundo elemento corresponde a una lista de tuplas con los **ratings predichos** por el algoritmo para las combinaciones usuarios e items presentes en el set de testeo, con la forma (user_id, item_id, rating_predicted).

In [ ]:
y_test, y_pred_users = sapling_similarity_users_model.test()
y_test, y_pred_items = sapling_similarity_items_model.test()

Además, los modelos entrenados cuentan con el método `predict`, que retorna para un `user_id` específico las lista de tuplas (item_id, ranking) de los items que el usuario no ha evaluado. Con estas predicciones es posible rankear una lista de items para un usuario ordenando descendente los items según el rating que predice el modelo.

In [ ]:
sapling_similarity_users_model.predict(user_id=1)[40:50]

[(1, np.int64(613), np.float64(4.300253965152188)),
 (1, np.int64(285), np.float64(4.293830614167815)),
 (1, np.int64(173), np.float64(4.290822452377635)),
 (1, np.int64(641), np.float64(4.2713802820914895)),
 (1, np.int64(1111), np.float64(4.26898936283368)),
 (1, np.int64(357), np.float64(4.265864353132638)),
 (1, np.int64(513), np.float64(4.262441584025546)),
 (1, np.int64(1431), np.float64(4.26102251250427)),
 (1, np.int64(657), np.float64(4.261015377303021)),
 (1, np.int64(479), np.float64(4.260759452916226))]

Se les entrega un diccionario con las predicciones hechas por cada una de las versiones del algoritmo, donde cada llave es un user_id y tiene como valor una lista con tuplas (item, rating), donde rating es el valor que el modelo predice que el usuario asignará al item. Los elementos para cada usuario estan ordenadas por el rating que predijo el modelo.

In [ ]:
user_based_predictions = {user_id: sapling_similarity_users_model.predict(user_id) for user_id in sapling_similarity_users_model.user_map.keys()}
item_based_predictions = {user_id: sapling_similarity_items_model.predict(user_id) for user_id in sapling_similarity_items_model.user_map.keys()}

1. Crea un diccionario con los items relevantes para cada usuario presentes en `y_test`. Cada llave del diccionario será un user_id y tendrá como valor un lista que contenga los item_ids de los items relevantes para ese usuario. Considere como relevante solo los items que se les asignó un **rating mayor o igual a 3**.

Ejemplo:
```python
{
  1: [3, 4, 8,...],
  2: [23, 24, 53,...],
  ...
}
```
donde [3, 4, 8,...] son los items relevantes para el usuario con ID = 1.

In [ ]:
user_relevant_items = {}

# COMPLETAR
### RESPUESTA ###
for user_id, item_id, rating in y_test:
    if user_id not in user_relevant_items:
        user_relevant_items[user_id] = []
    if rating >= 3:
        user_relevant_items[user_id].append(item_id)

2. Completa la función `user_auc`, la cual recibe como parámetros:

- `user_recommendations`:, Lista de tuplas (item_id, rating_predicted) de un usuario.

- `user_relevant_items` Lista de items_ids relevantes para el usuario.

- `k`: Largo de la lista de recomendación para calcular AUC.

Esta función deberá retornar el AUC para un usuario en particular. Basate en la siguiente fórmula para implementarlo:

$$
\text{AUC} =  \frac{1}{|I_u| \cdot |I \backslash I_u |} \sum_{(u,i,j) \in D_p} \delta(\hat{\phi}_{ui} \succ \hat{\phi}_{uj})
$$
donde:
- $|I_u|$: cantidad de items relevantes para el usuario $u$.
- $|I \backslash I_u|$: cantidad de items no relevantes para el usuario $u$.
- $(u,i,j)$: tripleta de usuario $u$, item relevante $i$ e item no relevante $j$.
- $D_p$: dataset para evaluación.
- $\delta(\hat{\phi}_{ui} \succ \hat{\phi}_{uj})$: Esta función vale 1 si el modelo ordenó correctamente un item relevante $i$ por encima de un item no relevante $j$ (es decir, si la predicción de $i$ es mayor a la de $j$). En caso contrario, vale 0.

---

**Ejemplo**

Supongamos que para un usuario tenemos:

`user_relevant_items = [A, B]`

`user_recommendations = [(A, 4.5), (C, 4.4), (B, 3.9), (D, 2.6)]`

Al comparar las tuplas de items (relevante, no_relevante) quedaría:
- (A, C): 4.5 > 4.4 → δ = 1
- (A, D): 4.5 > 2.6 → δ = 1
- (B, C): 3.9 > 4.4 → δ = 0 (mal ordenado)
- (B, D): 3.9 > 2.6 → δ = 1

Se tendría para dicho usuario un AUC de:
$$
\sum \delta = 3, AUC = \frac{3}{4} = 0.75
$$

In [ ]:
def user_auc(user_recommendations, user_relevant_items, k):
    # print(f"First {k} recommendations for user: {user_recommendations[:k]}")
    correct_items = 0
    relevant_items = []
    for item_id, rating in user_recommendations[:k]:
        if item_id in user_relevant_items:
            relevant_items.append((item_id, rating))
    # print(f"Relevant items for user: {relevant_items}")

    not_relevant_items = []
    for item_id, rating in user_recommendations[:k]:
        if item_id not in user_relevant_items:
            not_relevant_items.append((item_id, rating))
    # print(f"Not relevant items for user: {not_relevant_items}")

    for _, rating_relevant_item in relevant_items:
        for _, rating_not_relevant_item in not_relevant_items:
            if rating_relevant_item > rating_not_relevant_item:
                correct_items += 1

    return correct_items / (len(relevant_items) * len(not_relevant_items)) if relevant_items and not_relevant_items else 0

3. Llama a la función `average_auc` para cada modelo y compare usando valores de k de 10, 100, 500, 1.000 y 2.000. Analice los resultados y mencione por qué podría producirse la diferencia de valores.

In [ ]:
def average_auc(predictions_dict, k):
    """
    Calcula el AUC promedio dado un diccionario de predicciones y un tamaño k de recomendaciones.
    """
    total_auc_users = []

    for user_id, predictions in predictions_dict.items():
        if user_id in user_relevant_items:
            # Convertir tripletas (user_id, item_id, rating) a pares (item_id, rating)
            rec_list = []
            for _, item_id, rating in predictions:
                rec_list.append((int(item_id), float(rating)))

            auc_value = user_auc(rec_list, user_relevant_items[user_id], k)
            if auc_value > 0:
                total_auc_users.append(auc_value)

    return sum(total_auc_users) / len(total_auc_users) if total_auc_users else 0.0


In [ ]:
# COMPLETAR

# RESPUESTA
ks = [10, 100, 500, 1000, 2000]

for k in ks:
    user_auc_avg = average_auc(user_based_predictions, k)
    item_auc_avg = average_auc(item_based_predictions, k)
    print(f"k={k}: user-based AUC promedio={user_auc_avg:.4f}, item-based AUC promedio={item_auc_avg:.4f}")


k=10: user-based AUC promedio=0.3889, item-based AUC promedio=0.5490
k=100: user-based AUC promedio=0.4296, item-based AUC promedio=0.5166
k=500: user-based AUC promedio=0.5314, item-based AUC promedio=0.5984
k=1000: user-based AUC promedio=0.6491, item-based AUC promedio=0.7060
k=2000: user-based AUC promedio=0.7588, item-based AUC promedio=0.7792
